# News Data Processing — NASCAR Sponsor Mentions

Processes the manually tracked news data from Google Sheets into an analysis-ready dataset.

## Search Protocol (do this in Google Sheets / Google News first)

For each race + sponsor combination, use this search format in Google News:
```
"[Sponsor]" NASCAR "[Race Name]" 2024
```

**Examples:**
- `"FedEx" NASCAR "Daytona 500" 2024`
- `"NAPA" NASCAR "Coca-Cola 600" 2024`
- `"McDonald's" NASCAR "Ambetter Health 400" 2024`

**Search settings:**
1. Open Google in an incognito window
2. Go to Google News (news.google.com)
3. Set date range to the week after each race (race date + 7 days)
4. Record the first 10 results that contain actual sponsor mentions

**Mention types:**
- **Primary** — sponsor is main focus or in headline
- **Secondary** — sponsor mentioned significantly in body
- **Passing** — brief mention only (do NOT count these)

**Priority order for races:**
1. Daytona 500, Coca-Cola 600, Southern 500, Championship (major races)
2. All 10 playoff races
3. Races won by target drivers
4. Regular season races

Export your completed Google Sheet as `news_mentions_tracking.csv` and save to `data/raw/`.
A template is at `data/raw/news_mentions_tracking_template.csv`.

## Step 1: Set Up the News Tracking Template

In [7]:
import pandas as pd
import os

# Source quality tier mapping
source_to_tier = {
    'ESPN': 'Tier 1',
    'NASCAR.com': 'Tier 1',
    'Fox Sports': 'Tier 1',
    'CBS Sports': 'Tier 1',
    'NBC Sports': 'Tier 1',
    'Bleacher Report': 'Tier 2',
    'USA Today': 'Tier 2',
    'The Athletic': 'Tier 2',
    'SB Nation': 'Tier 2',
    'Yahoo Sports': 'Tier 3',
    'AP Sports': 'Tier 3',
    'Reuters': 'Tier 3',
}

tier_weights = {
    'Tier 1': 1.0,
    'Tier 2': 0.8,
    'Tier 3': 0.6,
    'Tier 4': 0.4,
}

tracking_path = '../data/raw/news_mentions_tracking.csv'

if not os.path.exists(tracking_path):
    print(f"\n*** File not found: {tracking_path}")
    print("Please complete the manual search process and save your tracking sheet.")
    print("Use the template at: ../data/raw/news_mentions_tracking_template.csv")
else:
    print(f"Found tracking file: {tracking_path}")

Found tracking file: ../data/raw/news_mentions_tracking.csv


## Step 2: Quality Control Checks

In [8]:
# Load and process news tracking data
news_df = pd.read_csv(tracking_path)

print(f"Total rows loaded: {len(news_df)}")
print(f"Columns: {list(news_df.columns)}")
print(f"\nSponsor breakdown:")
print(news_df['Sponsor'].value_counts())

# Completeness check — races covered by sponsor
print("\nRaces covered by sponsor:")
for sponsor in news_df['Sponsor'].unique():
    sponsor_races = news_df[news_df['Sponsor'] == sponsor]['Race_Number'].nunique()
    print(f"  {sponsor}: {sponsor_races}/36 races")

# Identify gaps
all_races = set(range(1, 37))
for sponsor in news_df['Sponsor'].unique():
    covered = set(news_df[news_df['Sponsor'] == sponsor]['Race_Number'].dropna().astype(int))
    missing = all_races - covered
    if missing:
        print(f"  {sponsor} missing races: {sorted(missing)}")

Total rows loaded: 107
Columns: ['Race_Number', 'Race_Name', 'Race_Date', 'Sponsor', 'Article_Date', 'Source', 'Headline', 'Article_URL', 'Mention_Type', 'Context', 'Verified']

Sponsor breakdown:
Sponsor
FedEx                  42
NAPA Auto Parts        39
McDonald's             14
Love's Travel Stops    12
Name: count, dtype: int64

Races covered by sponsor:
  FedEx: 35/36 races
  NAPA Auto Parts: 36/36 races
  McDonald's: 11/36 races
  Love's Travel Stops: 11/36 races
  FedEx missing races: [35, 36]
  NAPA Auto Parts missing races: [13]
  McDonald's missing races: [2, 3, 4, 7, 8, 9, 10, 11, 12, 13, 15, 16, 18, 19, 20, 21, 22, 25, 26, 29, 30, 31, 32, 33, 34, 36]
  Love's Travel Stops missing races: [4, 5, 6, 7, 8, 10, 11, 12, 13, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 28, 29, 30, 33, 34, 36]


In [9]:
# Duplicate check
duplicates = news_df.duplicated(subset=['Sponsor', 'Article_URL'])
if duplicates.sum() > 0:
    print(f"Warning: {duplicates.sum()} duplicate articles found")
    print(news_df[duplicates])
else:
    print("No duplicates found.")

# Source distribution
print("\nMention sources:")
print(news_df['Source'].value_counts().head(15))

print("\nMention types:")
print(news_df['Mention_Type'].value_counts())

    Race_Number                             Race_Name   Race_Date  \
10            2                   Ambetter Health 400  2024-02-25   
11            3                          Pennzoil 400  2024-03-03   
12            3                          Pennzoil 400  2024-03-03   
13            3                          Pennzoil 400  2024-03-03   
14            4               Shriners Children's 500  2024-03-10   
..          ...                                   ...         ...   
92           34             South Point 400 Las Vegas  2024-10-20   
93           35  Straight Talk Wireless 400 Homestead  2024-10-27   
94           35  Straight Talk Wireless 400 Homestead  2024-10-27   
95           35  Straight Talk Wireless 400 Homestead  2024-10-27   
96           36              Xfinity 500 Martinsville  2024-11-03   

                Sponsor Article_Date            Source  \
10  Love's Travel Stops   2024-02-26  TobyChristie.com   
11                FedEx   2024-03-04            Jayski 

## Step 3: Create Analysis-Ready Dataset

In [10]:
# Add source tier weights
news_df['Source_Tier'] = news_df['Source'].map(source_to_tier).fillna('Tier 3')
news_df['Tier_Weight'] = news_df['Source_Tier'].map(tier_weights)

# Filter to Primary and Secondary mentions only
news_df = news_df[news_df['Mention_Type'].isin(['Primary', 'Secondary'])]

print(f"Rows after filtering to Primary/Secondary: {len(news_df)}")

# Aggregate by sponsor and race
news_summary = news_df.groupby(['Sponsor', 'Race_Number', 'Race_Name']).agg(
    total_mentions=('Article_URL', 'count'),
    weighted_mentions=('Tier_Weight', 'sum'),
    primary_mentions=('Mention_Type', lambda x: (x == 'Primary').sum())
).reset_index()

# Rename columns to match deliverable schema
news_summary = news_summary.rename(columns={
    'Sponsor': 'sponsor',
    'Race_Number': 'race_number',
    'Race_Name': 'race_name',
})

print("\nNews Mentions Summary (sample):")
print(news_summary.head(15))

Rows after filtering to Primary/Secondary: 107

News Mentions Summary (sample):
   sponsor  race_number                       race_name  total_mentions  \
0    FedEx            1                     Daytona 500               2   
1    FedEx            2             Ambetter Health 400               1   
2    FedEx            3                    Pennzoil 400               1   
3    FedEx            4         Shriners Children's 500               1   
4    FedEx            5                   Food City 500               1   
5    FedEx            6        EchoPark Grand Prix COTA               1   
6    FedEx            7        Cookout 400 Martinsville               1   
7    FedEx            8   AutoTrader EchoPark 400 Texas               1   
8    FedEx            9             GEICO 500 Talladega               1   
9    FedEx           10         AdventHealth 400 Kansas               1   
10   FedEx           11                 Wurth 400 Dover               1   
11   FedEx          

## Step 4: Save Processed News Data

In [11]:
# Save the analysis-ready dataset
output_path = '../data/processed/news_mentions_2024.csv'
news_summary.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")

# Also save as final deliverable name
news_summary.to_csv('../data/processed/news_mentions.csv', index=False)
print("Saved final deliverable: news_mentions.csv")

# Save the raw filtered tracking data
news_df.to_csv('../data/raw/news_mentions_2024_raw.csv', index=False)
print("Saved raw tracking data")

print("\n=== News Data Processing Complete ===")

Saved to: ../data/processed/news_mentions_2024.csv
Saved final deliverable: news_mentions.csv
Saved raw tracking data

=== News Data Processing Complete ===


## Bonus Step: Final Validation (run after all three datasets are complete)

In [12]:
# Final validation script
print("=== Exposure Data Validation ===\n")

# Load all processed files
reddit  = pd.read_csv('../data/processed/reddit_mentions.csv')
youtube = pd.read_csv('../data/processed/youtube_engagement.csv')
news    = pd.read_csv('../data/processed/news_mentions.csv')

# Check sponsor coverage
target_sponsors = ['FedEx', 'NAPA Auto Parts', "McDonald's", "Love's Travel Stops"]

print("Sponsor Coverage:")
for sponsor in target_sponsors:
    r_count = len(reddit[reddit['sponsor'] == sponsor])
    y_count = len(youtube[youtube['sponsor'] == sponsor])
    n_count = len(news[news['sponsor'] == sponsor])
    print(f"  {sponsor}: Reddit={r_count}, YouTube={y_count}, News={n_count}")

# Check race coverage
print(f"\nRace Coverage:")
print(f"  Reddit:  {reddit['race_number'].nunique()}/36 races")
print(f"  YouTube: {youtube['race_number'].nunique()}/36 races")
print(f"  News:    {news['race_number'].nunique()}/36 races")

# Check for missing values
print(f"\nMissing Values:")
print(f"  Reddit:  {reddit.isnull().sum().sum()}")
print(f"  YouTube: {youtube.isnull().sum().sum()}")
print(f"  News:    {news.isnull().sum().sum()}")

# Summary statistics
print(f"\n=== Summary Statistics ===")
print(f"\nReddit:")
if 'mention_count' in reddit.columns:
    print(f"  Total mentions tracked: {reddit['mention_count'].sum():,}")
if 'total_score' in reddit.columns and 'total_comments' in reddit.columns:
    print(f"  Total engagement: {reddit['total_score'].sum() + reddit['total_comments'].sum():,}")

print(f"\nYouTube:")
if 'total_race_views' in youtube.columns:
    print(f"  Total views tracked: {youtube['total_race_views'].sum():,}")
if 'sponsor_video_views' in youtube.columns:
    print(f"  Sponsor-relevant views: {youtube['sponsor_video_views'].sum():,}")

print(f"\nNews:")
if 'total_mentions' in news.columns:
    print(f"  Total mentions: {news['total_mentions'].sum()}")
if 'primary_mentions' in news.columns:
    print(f"  Primary mentions: {news['primary_mentions'].sum()}")

=== Exposure Data Validation ===

Sponsor Coverage:
  FedEx: Reddit=26, YouTube=26, News=35
  NAPA Auto Parts: Reddit=26, YouTube=26, News=36
  McDonald's: Reddit=26, YouTube=26, News=11
  Love's Travel Stops: Reddit=26, YouTube=26, News=11

Race Coverage:
  Reddit:  26/36 races
  YouTube: 26/36 races
  News:    37/36 races

Missing Values:
  Reddit:  0
  YouTube: 0
  News:    0

=== Summary Statistics ===

Reddit:
  Total mentions tracked: 1,068
  Total engagement: 795

YouTube:
  Total views tracked: 1,433,844,976
  Sponsor-relevant views: 9,541,694

News:
  Total mentions: 107
  Primary mentions: 22
